# Notebook m4_a2_nb3_rag_itens_compras
Esse notebook exemplifica a criação de uma base de conhecimentos sobre itens de compras extraídos do PNCP

In [34]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [44]:
csv_file = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/compras/itens_compras.csv'



In [45]:
import pandas as pd

df_compras = pd.read_csv(csv_file, sep=";")
display(df_compras.head())

,ID_COMPRA,NUMERO_UASG,NUMERO_COMPRA,ANO_COMPRA,OBJETO,CHAVE_COMPRA_PNCP,ID_ITEM,numero_item,descricao,descricao_detalhada,unidade_fornecimento,valor_estimado,quantidade_solicitada,orcamento_sigiloso,codigo_item_catalogo,tipo_item_catalogo
0,696890,170209,68,2026,Fornecimento de materiais de cuidado dos cães ...,0039446000014110005372026,7773562,1,Vacina,"aplicação*: uso veterinário, forma farmacêutic...",Doses,132.50,3,N,439562,M
1,693455,927996,53,2026,Aquisição de MEDICAMENTOS/SUPLEMENTOS CONTROLA...,1695842500014810003442026,7717753,40,Fenobarbital Sódico,dosagem: 100 CONFORME DESCRITO DO TERMO DE REF...,Comprimido,0.07,509600,S,267660,M
2,693280,155124,34,2026,"Aquisição de Medicamentos Anti-inflamatórios, ...",1512643700030510029692026,7713479,11,Tramadol Cloridrato,dosagem: 50,Cápsula,0.15,1750,S,268534,M
3,693657,927502,356,2026,Aquisição dos medicamentos VACINA PARA IMUNOTE...,0073306200010210003352026,7719485,1,Vacina,"composição 1: extrato alérgeno de ácaros, form...",Frasco,281.22,10,N,483360,M
4,693455,927996,53,2026,Aquisição de MEDICAMENTOS/SUPLEMENTOS CONTROLA...,1695842500014810003442026,7717776,63,Levomepromazina,dosagem: 25 CONFORME DESCRITO DO TERMO DE REFE...,Comprimido,0.59,609700,S,268128,M


In [46]:
compras_json = df_compras.to_json(orient='records', lines=True, force_ascii=False).splitlines()

print(f"Total de {len(compras_json)} linhas JSON geradas.")
print("Primeiras 5 linhas JSON:")
for i, line in enumerate(compras_json[:5]):
    print(f"Linha {i+1}: {line}")

Total de 333 linhas JSON geradas.
Primeiras 5 linhas JSON:
Linha 1: {"ID_COMPRA":696890,"NUMERO_UASG":170209,"NUMERO_COMPRA":68,"ANO_COMPRA":2026,"OBJETO":"Fornecimento de materiais de cuidado dos cães de faro da RFB, como ração, vacinas, vermífugos, carrapaticidas, suplementos, medicamentos, vitaminas e materiais de higiene, conforme necessidade ALF\/AEG","CHAVE_COMPRA_PNCP":"0039446000014110005372026","ID_ITEM":7773562,"numero_item":1,"descricao":"Vacina","descricao_detalhada":"aplicação*: uso veterinário, forma farmacêutica: suspensão injetável, outros componentes: b. bronchiseptica, tipo: inativada ","unidade_fornecimento":"Doses","valor_estimado":132.5,"quantidade_solicitada":3,"orcamento_sigiloso":"N","codigo_item_catalogo":439562,"tipo_item_catalogo":"M"}
Linha 2: {"ID_COMPRA":693455,"NUMERO_UASG":927996,"NUMERO_COMPRA":53,"ANO_COMPRA":2026,"OBJETO":"Aquisição de MEDICAMENTOS\/SUPLEMENTOS CONTROLADOS PELA PORTARIA 344\/98 destinados as unidades de saúde pertencentes a Rede Hospi

## Criando uma Base de Conhecimento Vetorial

Para criar uma base de conhecimento vetorial, seguiremos estes passos:
1.  **Instalar bibliotecas necessárias**: `chromadb` e `openai`.
2.  **Gerar embeddings**: Converter esses pedaços em representações vetoriais numéricas usando o modelo de embedding da OpenAI.
3.  **Criar o armazenamento vetorial**: Armazenar esses embeddings em uma instância ChromaDB, que permite buscas eficientes por similaridade.

In [47]:
# Install necessary libraries for vector store and embeddings
!pip install -q chromadb openai

In [48]:
!pip install -q langchain
!pip install -q langchain-openai
!pip install -q langchain-community

In [49]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


In [50]:
env_path = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo IV - Grafos e IA/.env'

from dotenv import load_dotenv
load_dotenv(env_path)

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.documents import Document # Import Document class

# Initialize OpenAI Embeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

# Convert each JSON string in compras_json to a Document object
# Assuming each JSON string is a complete document, metadata can be added if available from the JSON.
documents_from_json = [Document(page_content=line) for line in compras_json]

# Check if 'db' (the vector database) already exists in memory and will be recreated.
if 'db' in globals() and db is not None:
    print("Base vetorial 'db' existente em memória detectada. Recriando...")

# Create the vector store using ChromaDB from the Document objects
db = Chroma.from_documents(documents_from_json, embeddings, persist_directory='vectordb')

print("Vector database created successfully from compras_json!")

Base vetorial 'db' existente em memória detectada. Recriando...
Vector database created successfully from compras_json!


A base de dados vetorial `db` está agora pronta. Você pode usá-la para buscas por similaridade ou para recuperar informações relevantes.

## Testando a Base Vetorial

Agora vamos testar a base vetorial recém-criada, realizando uma busca por similaridade com uma pergunta específica para recuperar o conteúdo relevante.

In [52]:
import json
# Definir a pergunta de consulta
query = "Existe na base alguma compra de Tramadol?"

def busca_semantica(query):

    # Realizar a busca por similaridade na base de dados vetorial
    # Recuperar os 15 chunks mais relevantes
    results = db.similarity_search(query, k=5)

    print(f"Recuperados {len(results)} chunks para a pergunta: '{query}'\n")

    # Apresentar os chunks recuperados
    for i, doc in enumerate(results):
        item = json.loads(doc.page_content)
        print(f"--- Chunk {i+1} ---")
        print(f'UASG: {item["NUMERO_UASG"]} ANO: {item["ANO_COMPRA"]} COMPRA: {item["NUMERO_COMPRA"]}')
        print(f'Objeto: {item["OBJETO"]}')
        print(f'Descrição: {item["descricao"]}')
        print(f'Descrição detalhada: {item["descricao_detalhada"]}')
        print(f'Unidade: {item["unidade_fornecimento"]}')


busca_semantica(query)

Recuperados 5 chunks para a pergunta: 'Existe na base alguma compra de Tramadol?'

--- Chunk 1 ---
UASG: 925449 ANO: 2026 COMPRA: 24
Objeto: Aquisição eventual de Medicamentos Sujeitos à Controle Especial orais e outros, para  atender a necessidade de 12 (doze) meses nas clínicas e unidades de terapia intensiva da  Fundação Pública Estadual Hospital de Clínicas Gaspar Vianna (FHCGV). 
Descrição: Tramadol Cloridrato
Descrição detalhada: dosagem: 50, forma farmacêutica: liberação lenta Tramadol  (cloridrato)  50mg 
Unidade: Cápsula
--- Chunk 2 ---
UASG: 157243 ANO: 2026 COMPRA: 35
Objeto: Medicamentos Anti-Inflamatórios, Analgésicos e Antitérmicos
Descrição: Tramadol Cloridrato
Descrição detalhada: dosagem: 50, forma farmacêutica: solução injetável 
Unidade: Ampola
--- Chunk 3 ---
UASG: 157243 ANO: 2026 COMPRA: 35
Objeto: Medicamentos Anti-Inflamatórios, Analgésicos e Antitérmicos
Descrição: Tramadol Cloridrato
Descrição detalhada: dosagem: 50, forma farmacêutica: solução injetável 
Unid

## Realizando Geração Aumentada por Recuperação (RAG)

Agora vamos integrar a base de conhecimento vetorial com um modelo de linguagem para realizar o RAG. Isso permitirá que o modelo de linguagem use o contexto recuperado dos seus documentos para gerar respostas mais precisas e informadas à sua pergunta.

In [54]:
from langchain_openai import ChatOpenAI
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain


# Initialize the LLM (Large Language Model)
# Using 'gpt-4o' or 'gpt-3.5-turbo' for better performance
llm = ChatOpenAI(model="gpt-5.4", temperature=0.1)

# Create a prompt template for RAG
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """Você é um especialista em compras de medicamentos.
                  O contexto fornecido corresponde as ultimas compras públicas de medicamentos realizadas pela Adm. Federal.
                  Responda à pergunta do usuário com base apenas no contexto fornecido:\n\n{context}
                  """),
    ("user", "{input}"),
])

# Create a chain that combines documents by 'stuffing' them into the prompt
document_chain = create_stuff_documents_chain(llm, rag_prompt)

# Create a retriever from your ChromaDB instance, returning 15 items
retriever = db.as_retriever(search_kwargs={'k': 15})

# Create the full retrieval chain
retrieval_chain = create_retrieval_chain(retriever, document_chain)



In [57]:
query = "quais os preços unitários do tramadol 50mg fornecido em ampolas? tente extrair para cada preço o volume da ampola"

# Invoke the chain with the test query
response = retrieval_chain.invoke({"input": query})

print("--- Resposta do RAG ---")
print(response["answer"])

# You can also inspect the retrieved documents that formed the context
print("\n--- Documentos de Contexto Recuperados ---")
for i, doc in enumerate(response["context"]):
    print(f"Documento {i+1} (Source: {doc.metadata.get('source', 'N/A')} - Page: {doc.metadata.get('page_label', 'N/A')}):\n{doc.page_content[:300]}...\n")

--- Resposta do RAG ---
Com base **apenas no contexto fornecido**, **não encontrei item de Tramadol 100 mg em ampola**.

Os registros em **ampola** que aparecem são de **Tramadol Cloridrato 50 mg / solução injetável**, com estes preços unitários:

| Compra | Descrição | Unidade | Preço unitário estimado | Volume da ampola |
|---|---|---:|---:|---:|
| UASG 110001, compra 90019/2026 | **50 mg/ml**, solução injetável | Ampola **2,00 ML** | **R$ 2,00** | **2,00 mL** |
| UASG 157243, compra 35/2026 | dosagem 50, solução injetável | Ampola | **R$ 1,24** | **não informado** |
| UASG 157243, compra 35/2026 | dosagem 50, solução injetável | Ampola | **R$ 1,20** | **não informado** |
| UASG 986411, compra 30/2026 | dosagem 50, solução injetável | Ampola | **R$ 0,89** | **não informado** |
| UASG 462314, compra 14/2026 | dosagem 50, solução injetável | Ampola | **R$ 1,99** | **não informado** |
| UASG 925448, compra 9/2026 | dosagem 50, solução injetável | Ampola | **R$ 1,43** | **não informado**

In [59]:
query = "quais os preços unitários do tramadol 100mg?"
# Invoke the chain with the test query
response = retrieval_chain.invoke({"input": query})

print("--- Resposta do RAG ---")
print(response["answer"])


--- Resposta do RAG ---
Com base no contexto fornecido, há **1 registro** de **Tramadol Cloridrato 100 mg**:

- **Comprimido** — **R$ 1,24 por unidade**  
  - Compra: **UASG 160366 / Pregão 90002/2026**
  - Quantidade solicitada: **5.500 comprimidos**

Se quiser, também posso listar os **preços do tramadol 50 mg** para comparação.
